# Opta Points

## Import libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [2]:
import warnings

import pandas as pd
from kloppy import statsbomb
from kloppy.domain import EventDataset, EventType
from config import paths, tournaments, players

In [3]:
warnings.filterwarnings(
    "ignore",
    message="The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Boolean Series key will be reindexed to match DataFrame index.",
    category=UserWarning,
)

## Load providers data

In [4]:
# StatsBomb match ID for UEFA Euro 2024 Final
match_ids = tournaments.get_all_match_ids(tournaments.EURO_2024)
EURO_2024_FINAL_MATCH_ID = match_ids[0]

In [5]:
# Load dataset based on provider
dataset = statsbomb.load(
    event_data=paths.STATSBOMB_EVENTS_DIR / f"{EURO_2024_FINAL_MATCH_ID}.json",
    lineup_data=paths.STATSBOMB_LINEUPS_DIR / f"{EURO_2024_FINAL_MATCH_ID}.json",
)

In [6]:
# Exclude events from penalty shootouts
filtered_dataset = dataset.filter(lambda event: event.period.id != 5)

## Convert dataset to DataFrame

In [7]:
# Define columns to extract
COLUMNS_TO_EXTRACT = [
    "player_id",
    "player",
    "team_id",
    "team",
    "event_id",
    "event_type",
    "result",
    "success",
    "body_part_type",
    "pass_type",
    "duel_type",
    "set_piece_type",
    "goalkeeper_type",
    "card_type",
    "coordinates_x",
    "coordinates_y",
    "time",
]

In [8]:
# Define data types for each column
DTYPE_MAPPING = {
    "player_id": "Int64",
    "player": "string",
    "team_id": "Int64",
    "team": "string",
    "event_id": "string",
    "event_type": "category",
    "result": "category",
    "success": "boolean",
    "body_part_type": "category",
    "pass_type": "category",
    "duel_type": "category",
    "set_piece_type": "category",
    "goalkeeper_type": "category",
    "card_type": "category",
    "coordinates_x": "Float64",
    "coordinates_y": "Float64",
    "time": "string",
}

In [9]:
# Convert dataset to DataFrame for easier exploration
df = filtered_dataset.to_df(*COLUMNS_TO_EXTRACT).astype(DTYPE_MAPPING)

In [10]:
# Display the first 5 rows of the dataset
df.head()

,player_id,player,team_id,team,event_id,event_type,result,success,body_part_type,pass_type,duel_type,set_piece_type,goalkeeper_type,card_type,coordinates_x,coordinates_y,time
0,<NA>,<NA>,772,Spain,50aa204f-5d65-4145-8597-5d5628fb7898,GENERIC:Starting XI,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
1,<NA>,<NA>,768,England,a279cbee-9ab3-4cfb-9c51-27cacc1bf2a2,GENERIC:Starting XI,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
2,<NA>,<NA>,768,England,d2126e70-9f04-4bb7-ba2b-9377836d1757,GENERIC:Half Start,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
3,<NA>,<NA>,772,Spain,54d78bfa-4146-42bd-acdc-97bcd393dd81,GENERIC:Half Start,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,P1T00:00
4,99174,Kobbie Mainoo,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,COMPLETE,True,RIGHT_FOOT,NaN,NaN,KICK_OFF,NaN,NaN,0.499564,0.499327,P1T00:00


In [11]:
# Display dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3415 entries, 0 to 3414
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   player_id        3400 non-null   Int64   
 1   player           3400 non-null   string  
 2   team_id          3415 non-null   Int64   
 3   team             3415 non-null   string  
 4   event_id         3415 non-null   string  
 5   event_type       3415 non-null   category
 6   result           1844 non-null   category
 7   success          1844 non-null   boolean 
 8   body_part_type   938 non-null    category
 9   pass_type        227 non-null    category
 10  duel_type        109 non-null    category
 11  set_piece_type   82 non-null     category
 12  goalkeeper_type  11 non-null     category
 13  card_type        8 non-null      category
 14  coordinates_x    3389 non-null   Float64 
 15  coordinates_y    3389 non-null   Float64 
 16  time             3415 non-null   string  


## Filter DataFrame to keep only events related to a player in player_info

In [12]:
# Get players match info
players_info = players.get_players_info(tournaments.EURO_2024_FINAL_MATCH_ID)

In [13]:
# Filter events to include only those involving players in players_info
players_df = df[df["player"].isin(players_info["player_name"])]

In [14]:
players_df.head()

,player_id,player,team_id,team,event_id,event_type,result,success,body_part_type,pass_type,duel_type,set_piece_type,goalkeeper_type,card_type,coordinates_x,coordinates_y,time
4,99174,Kobbie Mainoo,768,England,152820f0-6ca9-4df3-943b-a67d568ff472,PASS,COMPLETE,True,RIGHT_FOOT,NaN,NaN,KICK_OFF,NaN,NaN,0.499564,0.499327,P1T00:00
5,3468,Jordan Pickford,768,England,d64668c7-747c-4a7d-912c-e1c3ff357a67,GENERIC:Ball Receipt*,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,0.21834,0.48318,P1T00:03
6,3468,Jordan Pickford,768,England,9c107df3-a3c8-4ad5-bc35-00214087a105,CARRY,COMPLETE,True,NaN,NaN,NaN,NaN,NaN,NaN,0.21834,0.48318,P1T00:03
7,3468,Jordan Pickford,768,England,237201b8-aef8-4823-b282-e82875795c07,PASS,OUT,False,LEFT_FOOT,LONG_BALL,NaN,NaN,NaN,NaN,0.244381,0.386189,P1T00:05
8,22084,Bukayo Saka,768,England,c979e198-edc1-4f22-851a-26cedb6474cf,GENERIC:Ball Receipt*,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,0.798231,0.721654,P1T00:10


In [15]:
# Get players_df info
players_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3400 entries, 4 to 3412
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   player_id        3400 non-null   Int64   
 1   player           3400 non-null   string  
 2   team_id          3400 non-null   Int64   
 3   team             3400 non-null   string  
 4   event_id         3400 non-null   string  
 5   event_type       3400 non-null   category
 6   result           1844 non-null   category
 7   success          1844 non-null   boolean 
 8   body_part_type   938 non-null    category
 9   pass_type        227 non-null    category
 10  duel_type        109 non-null    category
 11  set_piece_type   82 non-null     category
 12  goalkeeper_type  11 non-null     category
 13  card_type        8 non-null      category
 14  coordinates_x    3387 non-null   Float64 
 15  coordinates_y    3387 non-null   Float64 
 16  time             3400 non-null   string  
dtype

## Extract player and team metrics

### Helper functions

In [16]:
def classify_players_by_position(dataset: EventDataset) -> dict[str, str]:
    """Determine if each player is a Field Player or Goalkeeper based on their positions."""
    player_position_dict = {}

    for team in dataset.metadata.teams:
        for player in team.players:
            for _, _, position in player.positions.ranges():
                if position.name == "Goalkeeper":  # Check if position is Goalkeeper
                    player_position_dict[player.name] = "Goalkeeper"
                    break
            if player.name not in player_position_dict:  # If not assigned as Goalkeeper, assign as Field Player
                player_position_dict[player.name] = "Field Player"

    return player_position_dict

In [17]:
def calculate_assists_for_player(player_data: pd.DataFrame, df: pd.DataFrame) -> int:
    """Calculate assists by looking for SHOT_ASSIST events before goals"""

    # Get all goal events in the match
    goals_df = df.loc[(df["event_type"] == "SHOT") & (df["result"] == "GOAL")]

    # Get player shot assist events
    shot_assists_df = player_data.loc[player_data["pass_type"] == "SHOT_ASSIST"]

    # Get indexes of DataFrames
    event_indexes = df.index.to_list()
    goal_indexes = goals_df.index.to_list()
    shot_assist_indexes = shot_assists_df.index.to_list()

    # Initialize assist counter
    assists_count = 0

    # Iterate through each shot assist and look for a goal in subsequent events
    for shot_assist_idx in shot_assist_indexes:
        event_idx = shot_assist_idx

        while event_idx in event_indexes:
            event_idx += 1
            event = df.iloc[event_idx]

            if event_idx in goal_indexes:  # Found a goal for this shot assist
                assists_count += 1
                break
            elif event["event_type"] == "SHOT":  # Found an unsuccessful shot
                break

    return assists_count

### Player metrics

In [18]:
ACTION_POINTS = {
    "goals": 10,
    "shots_on_target": 4,
    "shots_off_target": 2,
    "blocked_shots": 2,
    "own_goals": -5,
    "assists": 6,
    "passes": 0.2,
    "crosses": 0.2,
    "tackles": 2,
    "interceptions": 2,
    "fouls_won": 1,
    "fouls_conceded": -1,
    "offsides": -1,
    "yellow_cards": -2,
    "red_cards": -5,
    "goals_conceded_field_player": -1,
    "goals_conceded_goalkeeper": -6,
    "penalties_won": 4,
    "saves": 5,
    "penalties_saved": 5,
}

In [19]:
player_positions = classify_players_by_position(dataset)

In [20]:
def get_player_offsides(dataset: EventDataset, df: pd.DataFrame) -> dict[str, int]:
    """
    Calculate offsides for all players.

    Args:
        dataset: The event dataset.
        df: DataFrame containing event data.

    Returns:
        A dictionary mapping player names to their offside counts.
    """
    player_offsides = {}

    offside_event_ids = df.loc[(df["event_type"] == "PASS") & (df["result"] == "OFFSIDE"), "event_id"]

    for event_id in offside_event_ids:
        event = dataset.get_event_by_id(event_id)

        # Iterate through related events to find the receiver
        for related_event_id in event.related_event_ids:
            related_event = dataset.get_event_by_id(related_event_id)

            # Check if the related event is a GENERIC event (Ball Receipt)
            if related_event.event_type == EventType.GENERIC:
                receiver_name = related_event.player.name
                player_offsides[receiver_name] = player_offsides.get(receiver_name, 0) + 1

    return player_offsides

In [21]:
def get_team_conceded_goals(df: pd.DataFrame) -> dict[str, int]:
    """
    Calculate goals conceded for all teams.

    Args:
        df: DataFrame containing event data.

    Returns:
        A dictionary mapping team names to their goals conceded.
    """
    teams_goals_conceded = {}
    teams = df["team"].unique()

    for team in teams:
        goals_conceded = len(df.loc[(df["team"] != team) & (df["event_type"] == "SHOT") & (df["result"] == "GOAL")])
        teams_goals_conceded[team] = goals_conceded

    return teams_goals_conceded

In [22]:
def get_penalties_won(df: pd.DataFrame) -> dict[str, int]:
    """
    Calculate penalties won for all players.

    Args:
        df: DataFrame containing event data.

    Returns:
        A dictionary mapping player names to their penalties won.
    """
    penalties_won = {}

    # Get related events
    penalties_shot = df.loc[(df["event_type"] == "SHOT") & (df["set_piece_type"] == "PENALTY")]
    fouls_won = df.loc[(df["event_type"] == "GENERIC:Foul Won")]

    # Get indexes of DataFrames
    event_indexes = df.index.to_list()
    penalties_shot_indexes = penalties_shot.index.to_list()
    fouls_won_indexes = fouls_won.index.to_list()

    # Iterate through each penalty shot and look for a foul won in preceding events
    for penalty_shot_idx in penalties_shot_indexes:
        event_idx = penalty_shot_idx

        while event_idx in event_indexes:
            event_idx -= 1
            event = df.iloc[event_idx]

            if event_idx in fouls_won_indexes:  # Found a foul won for this penalty shot
                player_name = event["player"]
                penalties_won[player_name] = penalties_won.get(player_name, 0) + 1
                break
            elif event["event_type"] == "FOUL_COMMITTED	":  # No foul won found, stop searching
                break

    return penalties_won

In [23]:
def extract_player_metrics(
    df: pd.DataFrame,
    player_info_df: pd.DataFrame,
    player_positions: dict[str, str],
) -> dict[str, dict[str, str | int]]:
    """Extract player metrics needed for Opta Points calculation."""
    player_metrics = {}

    # Map player names to their teams and minutes played for quick access
    nickname_mapping = dict(zip(player_info_df["player_name"], player_info_df["nickname"]))
    team_mapping = dict(zip(player_info_df["player_name"], player_info_df["team_name"]))
    minutes_mapping = dict(zip(player_info_df["player_name"], player_info_df["minutes_played"]))

    # Execute helper functions
    offsides_mapping = get_player_offsides(dataset, df)
    team_goals_conceded = get_team_conceded_goals(df)
    penalties_won_mapping = get_penalties_won(df)

    for player_name in df["player"].unique():
        # Get specific player event data
        player_df = df.loc[df["player"] == player_name]

        # Use nickname, team and minutes mappings
        player_nickname = nickname_mapping[player_name] or player_name
        player_team = team_mapping[player_name]
        player_minutes = minutes_mapping[player_name]

        # Goals (G)
        goals = len(player_df.loc[(player_df["event_type"] == "SHOT") & (player_df["result"] == "GOAL")])

        # Shots on target (SonT)
        shots_on_target = len(
            player_df.loc[
                (player_df["event_type"] == "SHOT") & (player_df["result"].isin(["GOAL", "BLOCKED", "SAVED"]))
            ]
        )

        # Shots off target (SoffT)
        shots_off_target = len(
            player_df.loc[(player_df["event_type"] == "SHOT") & (player_df["result"].isin(["OFF_TARGET", "POST"]))]
        )

        # Blocked shots (BS)
        blocked_shots = len(player_df.loc[(player_df["event_type"] == "SHOT") & (player_df["result"] == "BLOCKED")])

        # Own goals (OG)
        own_goals = len(player_df.loc[(player_df["event_type"] == "SHOT") & (player_df["result"] == "OWN_GOAL")])

        # Assists (A)
        assists = calculate_assists_for_player(player_df, df)

        # Passes (P)
        passes = len(
            player_df.loc[
                (player_df["event_type"] == "PASS")
                & (player_df["success"])
                & (~player_df["pass_type"].isin(["CROSS", "HAND_PASS"]))
                & (player_df["set_piece_type"] != "THROW_IN")
            ]
        )

        # Crosses (C)
        crosses = len(
            player_df.loc[
                (player_df["event_type"] == "PASS") & (player_df["pass_type"] == "CROSS") & (player_df["success"])
            ]
        )

        # Tackles (Tk)
        tackles = len(
            player_df.loc[
                (player_df["event_type"] == "DUEL") & (player_df["duel_type"].isin(["TACKLE", "SLIDING_TACKLE"]))
            ]
        )

        # Interceptions (INT)
        interceptions = len(player_df.loc[(player_df["event_type"] == "INTERCEPTION") & (player_df["success"])])

        # Fouls won (FW)
        fouls_won = len(player_df.loc[(player_df["event_type"] == "GENERIC:Foul Won")])

        # Fouls conceded (FC)
        fouls_conceded = len(player_df.loc[(player_df["event_type"] == "FOUL_COMMITED")])

        # Offsides (O)
        offsides = offsides_mapping.get(player_name, 0)

        # Yellow cards (YC)
        yellow_cards = len(
            player_df.loc[(player_df["event_type"] == "CARD") & (player_df["card_type"] == "FIRST_YELLOW")]
        )

        # Red cards (RC)
        red_cards = len(
            player_df.loc[(player_df["event_type"] == "CARD") & (player_df["card_type"].isin(["SECOND_YELLOW", "RED"]))]
        )

        # Goals conceded (GC)
        goals_conceded = team_goals_conceded[player_team]

        # Penalties won (PW)
        # This is a subset of Fouls won (FW)
        penalties_won = penalties_won_mapping.get(player_name, 0)

        # Saves (SAV)
        saves = len(player_df.loc[(player_df["event_type"] == "GOALKEEPER") & (player_df["goalkeeper_type"] == "SAVE")])

        # Penalties saved (PS)
        # This is a subset of Saves (SAV)
        penalties_saved = len(
            player_df.loc[
                (player_df["event_type"] == "GOALKEEPER")
                & (player_df["goalkeeper_type"] == "SAVE")
                & (player_df["set_piece_type"] == "PENALTY")
            ]
        )

        player_metrics[player_nickname] = {
            "team": player_team,
            "position": player_positions[player_name],
            "minutes_played": player_minutes,
            "goals": goals,
            "shots_on_target": shots_on_target,
            "shots_off_target": shots_off_target,
            "blocked_shots": blocked_shots,
            "own_goals": own_goals,
            "assists": assists,
            "passes": passes,
            "crosses": crosses,
            "tackles": tackles,
            "interceptions": interceptions,
            "fouls_won": fouls_won,
            "fouls_conceded": fouls_conceded,
            "offsides": offsides,
            "yellow_cards": yellow_cards,
            "red_cards": red_cards,
            "goals_conceded": goals_conceded,
            "penalties_won": penalties_won,
            "saves": saves,
            "penalties_saved": penalties_saved,
        }

    return player_metrics

In [24]:
# Extract and display player metrics
player_metrics = extract_player_metrics(players_df, players_info, player_positions)
print(f"Player metrics extracted for {len(player_metrics)} players\n")

print("Player metrics:")
for k, v in player_metrics.items():
    print(f" -> {k}")
    print(f"    {v}")

Player metrics extracted for 29 players

Player metrics:
 -> Kobbie Mainoo
    {'team': 'England', 'position': 'Field Player', 'minutes_played': 71, 'goals': 0, 'shots_on_target': 0, 'shots_off_target': 0, 'blocked_shots': 0, 'own_goals': 0, 'assists': 0, 'passes': 14, 'crosses': 0, 'tackles': 0, 'interceptions': 1, 'fouls_won': 0, 'fouls_conceded': 0, 'offsides': 0, 'yellow_cards': 0, 'red_cards': 0, 'goals_conceded': 2, 'penalties_won': 0, 'saves': 0, 'penalties_saved': 0}
 -> Jordan Pickford
    {'team': 'England', 'position': 'Goalkeeper', 'minutes_played': 96, 'goals': 0, 'shots_on_target': 0, 'shots_off_target': 0, 'blocked_shots': 0, 'own_goals': 0, 'assists': 0, 'passes': 20, 'crosses': 0, 'tackles': 0, 'interceptions': 0, 'fouls_won': 0, 'fouls_conceded': 0, 'offsides': 0, 'yellow_cards': 0, 'red_cards': 0, 'goals_conceded': 2, 'penalties_won': 0, 'saves': 4, 'penalties_saved': 0}
 -> Bukayo Saka
    {'team': 'England', 'position': 'Field Player', 'minutes_played': 96, 'goals'

## Calculate the Opta Points for each player

In [25]:
def calculate_opta_points_for_players(player_metrics: dict) -> pd.DataFrame:
    """
    Calculate Opta Points for each player.

    Args:
        player_metrics: A dictionary containing player metrics.

    Returns:
        A DataFrame containing Opta Points for each player.
    """

    # Initialize list to hold player Opta Points
    opta_points_list = []

    for player, metrics in player_metrics.items():
        player_points = 0

        for metric, value in metrics.items():
            if metric in ACTION_POINTS:
                if metric == "goals_conceded":
                    if metrics["position"] == "Goalkeeper":
                        player_points += ACTION_POINTS["goals_conceded_goalkeeper"] * value
                    else:
                        player_points += ACTION_POINTS["goals_conceded_field_player"] * value
                else:
                    player_points += ACTION_POINTS[metric] * value

        # Append player Opta Points data to list
        opta_points_list.append(
            {
                "player": player,
                "team": metrics["team"],
                "opta_points": player_points,
                "minutes_played": metrics["minutes_played"],
                "goals": metrics["goals"],
                "assists": metrics["assists"],
                "passes": metrics["passes"],
            }
        )

    # Convert list to DataFrame
    opta_points_df = pd.DataFrame(opta_points_list)
    return opta_points_df

In [26]:
# Calculate PPI scores
opta_points_df = calculate_opta_points_for_players(player_metrics)

# Sort by Opta Points and rank players
opta_points_df = opta_points_df.sort_values("opta_points", ascending=False).reset_index(drop=True)
opta_points_df.insert(0, "rank", range(1, len(opta_points_df) + 1))

# Display all players ranked by Opta Points
display(opta_points_df)

,rank,player,team,opta_points,minutes_played,goals,assists,passes
0,1,Nico Williams,Spain,30.2,96,1,0,41
1,2,Declan Rice,England,25.4,96,0,0,32
2,3,Jordan Pickford,England,24.0,96,0,0,20
3,4,Aymeric Laporte,Spain,23.0,96,0,0,80
4,5,Mikel Oyarzabal,Spain,18.6,27,1,0,3
5,6,Fabián Ruiz,Spain,18.0,96,0,0,55
6,7,Robin Le Normand,Spain,18.0,84,0,0,80
7,8,Jude Bellingham,England,18.0,96,0,1,25
8,9,Lamine Yamal,Spain,17.6,90,0,1,23
9,10,Marc Cucurella,Spain,17.4,96,0,1,37
